# Chapter 06 — RAG pipelines · Practice

**Goal:** build a RAG pipeline one stage at a time on a small product-policy corpus (ingestion, chunking, embeddings, metadata filters, BM25, hybrid search with rank fusion, reranking, query rewriting, contextual retrieval, grounded answers with citations), measure each stage, and finish with one pipeline that answers through **Claude or OpenAI behind the same interface**.

- Theory: [README.md](README.md) (read section 6.x before running the matching section here)
- Estimated time: 2–3 hours
- Estimated API cost: **$0** offline. The optional live cells together cost **well under $0.05** (a few embeddings, one rerank call, one contextualizer call, one answer per vendor, one small vector store that the cell deletes again).
- Prerequisites: [00-prerequisites](../../00-prerequisites/README.md) (venv from `requirements.txt`; `.env` with `ANTHROPIC_API_KEY`, `OPENAI_API_KEY` and, for Voyage cells, `VOYAGE_API_KEY`)

| Marker | Meaning |
|---|---|
| **(offline)** | Runs with no key and no network. Model calls go through **real** `anthropic` / `openai` SDK clients whose HTTP layer is replaced by a small fake server, so request building and response parsing are the SDKs' own code. Only the model is fake. |
| **(live, costs money)** | Calls a real API. It runs only when `RUN_LIVE=1` is set in your environment **and** the key it needs is present; otherwise it prints "skipped". No offline cell depends on a live cell. |

The embedder and the reranker used offline are **toys**: tiny deterministic functions that imitate the one property each stage needs (embeddings match paraphrases but blur exact codes; a reranker reads the query and the chunk together). Real models do this far better; the optional cells show how to swap them in.

In [ ]:
import hashlib
import json
import math
import os
import re
from collections import Counter
from dataclasses import dataclass, field
from itertools import count
from typing import Callable

import anthropic
import httpx2
import openai
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel

load_dotenv(find_dotenv())
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
FAST_MODEL = "claude-haiku-4-5"
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
RUN_LIVE = os.getenv("RUN_LIVE", "0") == "1"      # live cells are opt-in


def live(*keys: str) -> bool:
    """True only when RUN_LIVE=1 and every named key is set; otherwise say why the cell is skipped."""
    missing = [k for k in keys if not os.getenv(k)]
    if RUN_LIVE and not missing:
        return True
    print("skipped (live, costs money): set RUN_LIVE=1" + (f" and {', '.join(missing)}" if missing else ""))
    return False


print("anthropic", anthropic.__version__, "| openai", openai.__version__, "| RUN_LIVE =", RUN_LIVE)

### Offline toolkit (run this once)

- `tokenize()` lowercases, keeps codes such as `e-305` whole, drops stop words and strips a plural `s`. BM25 and the metrics use it.
- `fake_claude(handler)` and `fake_openai(handler)` return **real** SDK clients. Each request body goes to your `handler`, which returns the response body. `CALLS` records every request.

In [ ]:
STOP = set("""a an the and or of to in on for with is are be by it its this that i my me you your we our us do does
did can could how what when where if at as from not no will would under over after before about which who than
then there their they them has have had was were been any all per should get once""".split())


def stem(t: str) -> str:
    if len(t) > 4 and t.endswith("ies"):
        return t[:-3] + "y"
    if len(t) > 3 and t.endswith("s") and not t.endswith("ss"):
        return t[:-1]
    return t


def tokenize(text: str) -> list[str]:
    return [stem(t) for t in re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower()) if t not in STOP]


def slug(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", s.lower()).strip("-")


CALLS: list[dict] = []
_ids = count(1)


def _mock(handler):
    def handle(request: httpx2.Request) -> httpx2.Response:
        body = json.loads(request.content)
        CALLS.append({"url": str(request.url), "body": body})
        return httpx2.Response(200, json=handler(str(request.url), body))
    return httpx2.MockTransport(handle)


def fake_claude(handler) -> anthropic.Anthropic:
    return anthropic.Anthropic(api_key="offline", max_retries=0,
                               http_client=anthropic.DefaultHttpxClient(transport=_mock(handler)))


def fake_openai(handler) -> openai.OpenAI:
    return openai.OpenAI(api_key="offline", max_retries=0,
                         http_client=openai.DefaultHttpxClient(transport=_mock(handler)))


def claude_reply(blocks: list[dict], usage: dict | None = None) -> dict:
    """A Messages API response body (wire format)."""
    return {"id": f"msg_{next(_ids):04d}", "type": "message", "role": "assistant", "model": "fake",
            "content": blocks, "stop_reason": "end_turn", "stop_sequence": None,
            "usage": usage or {"input_tokens": 100, "output_tokens": 30}}


print("toolkit ready:", tokenize("The Ember stove shows E-305 errors on 2 days"))

## 6.1–6.2 The corpus and ingestion

The corpus: policies and owner's guides of a fictional outdoor-gear shop. Each document is Markdown with **metadata** decided at ingestion time: `version`, `is_current`, `region`, `group` (who may read it) and a `content_hash`. Note the traps on purpose: an **old version** of the returns policy, an **EU-only** section, an **internal** staff document, and two owner's guides whose warranty sections never name their product. [Theory](README.md#62-ingestion-parse-clean-add-metadata)

**Do we need RAG at all?** This toy corpus is tiny, so in real life you would put it all in the prompt ([6.1](README.md#61-what-rag-is-and-when-you-need-it)). We use it because every retrieval failure stays visible at this size.

In [ ]:
RAW_DOCS = [
    {"doc_id": "returns", "version": 3, "is_current": True, "region": "all", "group": "public", "text": """
# Returns and refunds policy
## Standard returns
You can return unused items within 30 days of delivery for a full refund to the original payment method. Items must include their original tags.
## Opened or used items
Opened items can be returned within 14 days for store credit, not a cash refund. Used items are not accepted unless they are faulty.
## Refund timing
Refunds are issued within 5 business days after the warehouse receives the item. Card refunds can take a further 3 to 7 days to appear on a statement.
"""},
    {"doc_id": "returns-eu", "version": 3, "is_current": True, "region": "EU", "group": "public", "text": """
# Returns policy for EU customers
## Right of withdrawal
Customers in the EU can withdraw from a purchase within 14 days of delivery without giving a reason, including opened items, and receive a full refund.
"""},
    {"doc_id": "returns-v2", "version": 2, "is_current": False, "region": "all", "group": "public", "text": """
# Returns and refunds policy
## Standard returns
You can return unused items within 60 days of delivery for a full refund.
"""},
    {"doc_id": "shipping", "version": 1, "is_current": True, "region": "all", "group": "public", "text": """
# Shipping policy
## Delivery times
Standard delivery takes 3 to 5 business days. Express delivery takes 1 to 2 business days and costs $12.
## Free shipping
Orders over $75 ship free with standard delivery.
## Lost parcels
If tracking shows no movement for 7 days, contact support and we will resend or refund the order.
"""},
    {"doc_id": "trailhead-2", "version": 1, "is_current": True, "region": "all", "group": "public", "text": """
# Trailhead 2 tent owner's guide
## Setup
Clip the inner tent to the poles before you attach the fly sheet. Stake the corners first on windy days.
## Warranty
The frame and poles are covered for 5 years against manufacturing defects. Fabric and zips are covered for 2 years.
## Troubleshooting
If the hub connector cracks, stop using the frame and order replacement hub HX-40 from support.
"""},
    {"doc_id": "summit-40", "version": 1, "is_current": True, "region": "all", "group": "public", "text": """
# Summit 40 backpack owner's guide
## Fit
Loosen all straps, put the pack on and tighten the hip belt first so it carries most of the weight.
## Warranty
The frame sheet is covered for 2 years against manufacturing defects. Buckles and straps are covered for 1 year.
## Care
Hand wash with mild soap and let it dry in the shade. Do not machine wash.
"""},
    {"doc_id": "ember", "version": 1, "is_current": True, "region": "all", "group": "public", "text": """
# Ember camp stove owner's guide
## Error E-217
E-217 means the igniter is not sparking. Check the battery and clean the electrode with a dry cloth.
## Error E-305
E-305 means the fuel valve is blocked. Let the stove cool down, then clear the valve with the pin tool.
## Safety
Never use the stove inside a tent or a closed vehicle, because carbon monoxide can build up.
"""},
    {"doc_id": "warranty-claims", "version": 1, "is_current": True, "region": "all", "group": "public", "text": """
# Warranty claims
## How to claim
Send a photo of the defect and your order number through the support form. Replacement parts are sent at no cost.
## What is not covered
Damage from misuse, normal wear and tear, or repairs by third parties is not covered.
"""},
    {"doc_id": "staff-perks", "version": 1, "is_current": True, "region": "all", "group": "staff", "text": """
# Staff perks (internal)
## Employee discount
Employees get 40% off full-price items, up to $2,000 per year.
"""},
]


def ingest(raw: dict) -> dict:
    """Clean the text and attach the metadata every later stage relies on."""
    text = re.sub(r"[ \t]+", " ", raw["text"]).strip()
    title = text.splitlines()[0].lstrip("# ").strip()
    meta = {k: raw[k] for k in ("version", "is_current", "region", "group")}
    meta |= {"title": title, "source": f"kb://{raw['doc_id']}?v={raw['version']}",
             "content_hash": hashlib.sha256(text.encode()).hexdigest()[:12]}
    return {"doc_id": raw["doc_id"], "text": text, "meta": meta}


DOCS = {d["doc_id"]: d for d in map(ingest, RAW_DOCS)}
for d in DOCS.values():
    m = d["meta"]
    print(f"{d['doc_id']:16} v{m['version']} current={m['is_current']!s:5} region={m['region']:3} "
          f"group={m['group']:6} hash={m['content_hash']}")

> **Observe:** `content_hash` is what makes incremental re-indexing cheap: on the next sync, only documents whose hash changed are re-chunked and re-embedded. The old `returns-v2` stays in the store but is marked `is_current=False`, so a filter (6.5) can hide it.

## 6.3 Chunking strategies

Two chunkers on the same document: fixed-size windows with overlap, and the README's structure-aware chunker (one chunk per section, long sections split with 20% overlap, heading path kept in metadata). [Theory](README.md#63-chunking-strategies)

In [ ]:
@dataclass
class Chunk:
    id: str
    doc_id: str
    text: str
    meta: dict = field(default_factory=dict)


def chunk_fixed(doc_id: str, text: str, size: int = 18, overlap: int = 4) -> list[Chunk]:
    words, step = text.split(), size - overlap
    return [Chunk(f"{doc_id}#w{i}", doc_id, " ".join(words[i:i + size]))
            for i in range(0, len(words), step) if words[i:i + size]]


def chunk_markdown(doc_id: str, text: str, max_words: int = 150) -> list[Chunk]:
    """One chunk per section; split long sections with 20% overlap (the README's chunker)."""
    chunks, title, section = [], "", ""
    for block in re.split(r"\n(?=#)", text.strip()):
        heading, _, body = block.partition("\n")
        if heading.startswith("# "):
            title = heading[2:]
        section = heading.lstrip("# ")
        words, step = body.split(), int(max_words * 0.8)
        for n, start in enumerate(range(0, len(words), step)):
            chunks.append(Chunk(id=f"{doc_id}#{slug(section)}-{n}", doc_id=doc_id,
                                text=" ".join(words[start:start + max_words]),
                                meta={"title": title, "section": section}))
            if start + max_words >= len(words):
                break                     # this window reached the section's end
    return chunks


for c in chunk_fixed("ember", DOCS["ember"]["text"])[:3]:
    print("fixed     |", c.text)
print()
for c in chunk_markdown("ember", DOCS["ember"]["text"]):
    print("structure |", c.id, "|", c.text[:70])

> **Observe:** the fixed windows mix headings into the text, split "carbon monoxide can build up" away from the warning it belongs to, and put two error codes in one chunk. The structure-aware chunks match the document's own units, and each one carries its section in metadata for filtering and citations.

Now chunk the whole corpus and copy the document metadata onto every chunk (filters run on chunks, not documents).

In [ ]:
CHUNKS: list[Chunk] = []
for d in DOCS.values():
    for c in chunk_markdown(d["doc_id"], d["text"]):
        c.meta = {**d["meta"], **c.meta}          # document metadata + section
        CHUNKS.append(c)
BY_ID = {c.id: c for c in CHUNKS}
print(len(CHUNKS), "chunks, e.g.", CHUNKS[0].id, CHUNKS[0].meta)

## 6.4 Embeddings and vector indexes

`ToyEmbedder` stands in for an embedding model. It maps words to shared **concepts** ("money", "refund" and "reimburse" all become `refund`), hashes the concepts into a 256-dimension vector and normalizes it. Like a real dense model, it matches paraphrases; unlike BM25 it **ignores tokens with digits**, which imitates how dense models blur exact codes such as `E-305`. `DenseIndex` is a flat (exact) index: it compares the query with every vector. [Theory](README.md#64-embeddings-and-vector-indexes)

In [ ]:
CONCEPTS = {
    "money": "refund", "reimburse": "refund", "repay": "refund", "refund": "refund",
    "wore": "used", "worn": "used", "wear": "used", "use": "used", "used": "used",
    "open": "opened", "opened": "opened", "unboxed": "opened",
    "broken": "defect", "faulty": "defect", "defect": "defect", "crack": "defect", "cracked": "defect",
    "damage": "defect", "guarantee": "warranty", "warranty": "warranty", "covered": "warranty",
    "cover": "warranty", "delivery": "ship", "shipping": "ship", "ship": "ship", "parcel": "parcel",
    "package": "parcel", "charge": "price", "cost": "price", "price": "price", "fee": "price",
    "burner": "stove", "cooker": "stove", "cook": "stove", "shelter": "tent", "backpack": "pack",
    "rucksack": "pack", "bag": "pack", "spark": "ignite", "sparking": "ignite", "igniter": "ignite",
    "light": "ignite", "gas": "fuel", "clogged": "blocked", "employee": "staff", "europe": "eu",
    "european": "eu", "laundry": "wash", "card": "payment", "statement": "payment", "quickly": "time",
    "long": "time", "moved": "movement", "week": "day",
}


class ToyEmbedder:
    dim = 256

    def _vec(self, text: str) -> list[float]:
        v = [0.0] * self.dim
        for t in tokenize(text):
            if any(ch.isdigit() for ch in t):
                continue                      # dense models blur exact codes; the toy drops them
            concept = CONCEPTS.get(t, t)
            v[int(hashlib.md5(concept.encode()).hexdigest(), 16) % self.dim] += 1.0
        norm = math.sqrt(sum(x * x for x in v)) or 1.0
        return [x / norm for x in v]

    def embed(self, texts: list[str], input_type: str = "document") -> list[list[float]]:
        return [self._vec(t) for t in texts]


def cosine(a: list[float], b: list[float]) -> float:
    return sum(x * y for x, y in zip(a, b))       # vectors are normalized


def text_plain(c: Chunk) -> str:
    return c.text


class DenseIndex:
    """Flat index: exact cosine against every vector. `where` filters inside the search (6.5)."""

    def __init__(self, chunks, embedder, text_of: Callable[[Chunk], str] = text_plain):
        self.chunks, self.embedder = chunks, embedder
        self.vectors = embedder.embed([text_of(c) for c in chunks], input_type="document")

    def search(self, query: str, k: int = 20, where=None) -> list[Chunk]:
        q = self.embedder.embed([query], input_type="query")[0]
        pool = [(c, v) for c, v in zip(self.chunks, self.vectors) if where is None or where(c)]
        ranked = sorted(pool, key=lambda cv: -cosine(q, cv[1]))
        return [c for c, _ in ranked[:k]]


dense = DenseIndex(CHUNKS, ToyEmbedder())
for q in ["Can I get my money back for a jacket I wore once?", "What does error E-305 mean on the Ember stove?"]:
    print(q, "->", [c.id for c in dense.search(q, k=3)])

> **Observe:** the paraphrase works (no chunk contains "money back" or "wore", yet the opened-or-used section comes first), but the error-code question puts an unrelated chunk first. The toy drops `E-305`, so it can only match generic words such as "mean" and "stove". Remember this; BM25 fixes it in 6.6.

### Real embedding models (offline request check, then live)

The same `embed()` interface over OpenAI's SDK. Offline, a fake server returns toy vectors, so you can see the exact request the SDK sends. The live cell calls OpenAI and Voyage for real.

In [ ]:
class OpenAIEmbedder:
    def __init__(self, client: openai.OpenAI, model: str = "text-embedding-3-small", dims: int = 512):
        self.client, self.model, self.dims = client, model, dims

    def embed(self, texts: list[str], input_type: str = "document") -> list[list[float]]:
        r = self.client.embeddings.create(model=self.model, input=texts, dimensions=self.dims)
        return [d.embedding for d in sorted(r.data, key=lambda d: d.index)]


class VoyageEmbedder:
    """Voyage over plain HTTP (Anthropic's docs point to Voyage; there is no Claude embedding model)."""
    url = "https://ai.mongodb.com/v1/embeddings"

    def __init__(self, model: str = "voyage-4"):
        self.model = model

    def embed(self, texts: list[str], input_type: str = "document") -> list[list[float]]:
        r = httpx2.post(self.url, headers={"Authorization": f"Bearer {os.environ['VOYAGE_API_KEY']}"},
                        json={"input": texts, "model": self.model, "input_type": input_type}, timeout=60)
        r.raise_for_status()
        return [d["embedding"] for d in sorted(r.json()["data"], key=lambda d: d["index"])]


def embeddings_server(url: str, body: dict) -> dict:
    toy = ToyEmbedder().embed(body["input"])
    return {"object": "list", "model": body["model"], "usage": {"prompt_tokens": 9, "total_tokens": 9},
            "data": [{"object": "embedding", "index": i, "embedding": v} for i, v in enumerate(toy)]}


CALLS.clear()
oai_dense = DenseIndex(CHUNKS, OpenAIEmbedder(fake_openai(embeddings_server)))
print(json.dumps({k: (v if k != "input" else v[:2]) for k, v in CALLS[0]["body"].items()}, indent=1))
print(len(CALLS), "request(s) to", CALLS[0]["url"])

In [ ]:
# (live, costs money: well under $0.01) the same index with real embedding models
if live("OPENAI_API_KEY"):
    real = DenseIndex(CHUNKS, OpenAIEmbedder(openai.OpenAI()))
    print("OpenAI :", [c.id for c in real.search("What does error E-305 mean on the Ember stove?", k=3)])
if os.getenv("VOYAGE_API_KEY") and live("VOYAGE_API_KEY"):
    real = DenseIndex(CHUNKS, VoyageEmbedder())
    print("Voyage :", [c.id for c in real.search("What does error E-305 mean on the Ember stove?", k=3)])

> **Try it:** if `fastembed` from `requirements-extras.txt` is installed, write a `FastEmbedder` with the same `embed()` method (`TextEmbedding().embed(texts)` returns NumPy vectors; convert with `.tolist()`) and compare its rankings with the toy's. Never mix vectors from two models in one index: re-embed everything.

## 6.5 Indexing strategies: filter inside the search

Access rules are filters **inside** the search. Compare a customer in the US asking about returns: ranking first and filtering afterwards versus filtering inside the search. [Theory](README.md#65-indexing-strategies)

In [ ]:
@dataclass
class User:
    region: str
    groups: set


def allowed_for(user: User):
    return lambda c: (c.meta["is_current"] and c.meta["group"] in user.groups
                      and c.meta["region"] in ("all", user.region))


customer_us = User(region="US", groups={"public"})
q = "What is the time limit to return items for a refund?"

ranked_then_filtered = [c for c in dense.search(q, k=4) if allowed_for(customer_us)(c)]
filtered_inside = dense.search(q, k=4, where=allowed_for(customer_us))
print("rank, then filter :", [c.id for c in ranked_then_filtered])
print("filter inside     :", [c.id for c in filtered_inside])
print("unfiltered top 4  :", [c.id for c in dense.search(q, k=4)])

> **Observe:** the unfiltered top 4 starts with the superseded 60-day policy. Filtering afterwards leaves only 3 results; filtering inside the search always fills the top k with chunks this user may see. In production the vector database applies the filter (a `WHERE` clause in pgvector, metadata filters in a managed store, `filters` in OpenAI `file_search`).

## 6.6 BM25 lexical search

The README's `BM25` class, with the same `where` filter. Then a small labeled question set and two metrics: **recall@k** (is the right chunk in the top k?) and **MRR** (mean of 1/rank of the right chunk). [Theory](README.md#66-bm25-lexical-search)

In [ ]:
class BM25:
    def __init__(self, chunks, k1=1.5, b=0.75, text_of: Callable[[Chunk], str] = text_plain):
        self.chunks, self.k1, self.b = chunks, k1, b
        self.docs = [tokenize(text_of(c)) for c in chunks]
        self.avg = sum(map(len, self.docs)) / len(self.docs)
        n, N = Counter(t for d in self.docs for t in set(d)), len(self.docs)
        self.idf = {t: math.log(1 + (N - f + 0.5) / (f + 0.5)) for t, f in n.items()}

    def score(self, q: list[str], d: list[str]) -> float:
        tf, norm = Counter(d), self.k1 * (1 - self.b + self.b * len(d) / self.avg)
        return sum(self.idf.get(t, 0) * tf[t] * (self.k1 + 1) / (tf[t] + norm) for t in q if tf[t])

    def search(self, query: str, k: int = 20, where=None) -> list[Chunk]:
        q = tokenize(query)
        scored = [(c, self.score(q, d)) for c, d in zip(self.chunks, self.docs) if where is None or where(c)]
        ranked = sorted(scored, key=lambda p: -p[1])
        return [c for c, s in ranked[:k] if s > 0]


bm25 = BM25(CHUNKS)
print([c.id for c in bm25.search("What does error E-305 mean on the Ember stove?", k=3)])

In [ ]:
EVAL = [   # (question, id of the chunk that answers it)
    ("Can I get my money back for a jacket I wore once?", "returns#opened-or-used-items-0"),
    ("What does error E-305 mean on the Ember stove?", "ember#error-e-305-0"),
    ("My Ember stove shows E-217, what should I check?", "ember#error-e-217-0"),
    ("How long is the Summit 40 frame covered?", "summit-40#warranty-0"),
    ("How long are the Trailhead tent poles under guarantee?", "trailhead-2#warranty-0"),
    ("My parcel hasn't moved for over a week, what now?", "shipping#lost-parcels-0"),
    ("Do you charge for delivery on a $90 order?", "shipping#free-shipping-0"),
    ("Can I cook inside my tent with the stove?", "ember#safety-0"),
    ("Which part do I need if the hub connector cracks?", "trailhead-2#troubleshooting-0"),
    ("How quickly will a refund reach my card?", "returns#refund-timing-0"),
    ("What damage does the warranty not cover?", "warranty-claims#what-is-not-covered-0"),
    ("How do I wash the Summit 40 pack?", "summit-40#care-0"),
]
PUBLIC_US = allowed_for(customer_us)


def evaluate(search: Callable[[str], list[Chunk]], k: int = 3) -> dict:
    ranks = []
    for question, gold in EVAL:
        ids = [c.id for c in search(question)]
        ranks.append(ids.index(gold) + 1 if gold in ids else None)
    return {"recall@1": sum(r == 1 for r in ranks) / len(ranks),
            f"recall@{k}": sum(r is not None and r <= k for r in ranks) / len(ranks),
            "mrr": sum(1 / r for r in ranks if r) / len(ranks), "ranks": ranks}


RESULTS = {}


def report(name: str, search) -> None:
    RESULTS[name] = r = evaluate(search)
    print(f"{name:28} recall@1={r['recall@1']:.2f} recall@3={r['recall@3']:.2f} mrr={r['mrr']:.2f}  ranks={r['ranks']}")


report("dense (toy embeddings)", lambda q: dense.search(q, k=20, where=PUBLIC_US))
report("BM25", lambda q: bm25.search(q, k=20, where=PUBLIC_US))

> **Observe:** read the `ranks` lists question by question (`None` = not retrieved at all). BM25 puts both error codes at rank 1 but misses two questions completely: "money back ... wore" shares no word with the answer, and the lost-parcels answer never says "parcel" (that word sits in the section heading, which is not part of the chunk text). The dense side finds every answer somewhere in the top 3 but ranks both error-code questions and the Summit 40 warranty question second. Neither retriever is good enough alone.

## 6.7 Hybrid search: a multi-index pipeline with rank fusion

Run both indexes and fuse the ranked lists with reciprocal rank fusion. [Theory](README.md#67-hybrid-search-a-multi-index-pipeline-with-rank-fusion)

In [ ]:
def rrf(rankings: list[list[str]], k: int = 60, weights: list[float] | None = None) -> list[str]:
    """Fuse ranked lists of chunk ids; higher fused score ranks first."""
    weights = weights or [1.0] * len(rankings)
    scores: dict[str, float] = {}
    for w, ranking in zip(weights, rankings):
        for rank, cid in enumerate(ranking, start=1):
            scores[cid] = scores.get(cid, 0.0) + w / (k + rank)
    return sorted(scores, key=lambda cid: -scores[cid])


def hybrid_search(dense_index, bm25_index, query: str, k: int = 20, where=None) -> list[Chunk]:
    fused = rrf([[c.id for c in dense_index.search(query, k=50, where=where)],
                 [c.id for c in bm25_index.search(query, k=50, where=where)]])
    return [BY_ID[cid] for cid in fused[:k]]


report("hybrid (RRF)", lambda q: hybrid_search(dense, bm25, q, where=PUBLIC_US))

> **Observe:** hybrid beats both retrievers on recall@1 and MRR, because each one rescues the other's misses (BM25's two `None`s are back at rank 1). The top-1 order is still imperfect: RRF knows ranks, not meaning. That is the reranker's job.

> **Try it:** pass `weights=[1.0, 2.0]` inside `hybrid_search` (lean toward BM25) and rerun. Which questions move, and why would a parts-catalog assistant want that weighting while a policy assistant might not?

## 6.8 Reranking

`toy_rerank` stands in for a cross-encoder: it reads the query **and** each candidate together and scores how much of the query the chunk answers, giving credit for both exact terms and concept matches, weighted by rarity. Retrieve wide (top 20 fused), rerank, keep the top 5. [Theory](README.md#68-reranking)

In [ ]:
def toy_rerank(query: str, chunks: list[Chunk], top_k: int = 5,
               text_of: Callable[[Chunk], str] = text_plain) -> list[tuple[Chunk, float]]:
    """Score in [0, 1]: rarity-weighted share of query terms the chunk covers (exactly or by concept)."""
    q = tokenize(query)
    idf = bm25.idf

    def covered(t: str, toks: list[str]) -> float:
        if t in toks:
            return 1.0
        return 0.9 if t in CONCEPTS and CONCEPTS[t] in {CONCEPTS.get(x, x) for x in toks} else 0.0

    scored = []
    for c in chunks:
        toks = tokenize(text_of(c))
        weights = [idf.get(t, 2.5) for t in q]
        hit = sum(w * covered(t, toks) for t, w in zip(q, weights))
        scored.append((c, round(hit / (sum(weights) or 1.0), 3)))
    return sorted(scored, key=lambda p: -p[1])[:top_k]


def hybrid_rerank(question: str, where=PUBLIC_US) -> list[Chunk]:
    return [c for c, _ in toy_rerank(question, hybrid_search(dense, bm25, question, k=20, where=where))]


report("hybrid + rerank", hybrid_rerank)

In [ ]:
# (live, costs money: well under $0.01) a real cross-encoder: Voyage rerank-3 over the fused top 20
if live("VOYAGE_API_KEY"):
    question = "How long is the Summit 40 frame covered?"
    candidates = hybrid_search(dense, bm25, question, k=20, where=PUBLIC_US)
    r = httpx2.post("https://ai.mongodb.com/v1/rerank",
                    headers={"Authorization": f"Bearer {os.environ['VOYAGE_API_KEY']}"},
                    json={"query": question, "documents": [c.text for c in candidates], "model": "rerank-3", "top_k": 3},
                    timeout=60)
    r.raise_for_status()
    for d in r.json()["data"]:
        print(round(d["relevance_score"], 3), candidates[d["index"]].id)

> **Observe:** reranking fixes the E-217 and tent-poles questions. Two stay wrong. "Do you charge for delivery on a $90 order?" needs a comparison ($90 is over the $75 threshold) that word matching cannot do; a real cross-encoder, or the generator reading the top 5, handles it. "How long is the Summit 40 frame covered?" is a tie: the Summit and Trailhead warranty chunks both say "the frame ... is covered for N years" and neither names its product, so **no retriever and no reranker** can tell them apart from the chunk text. That is a chunking-context problem, fixed in 6.10.

## 6.9 Query transformation

In a chat, the last message is often not a usable query. A fast model rewrites it into a standalone query (here with structured output through `messages.parse`; the fake plays the model). [Theory](README.md#69-query-transformation)

In [ ]:
class Rewrite(BaseModel):
    standalone_query: str
    alternatives: list[str]


REWRITE_SYSTEM = ("Rewrite the user's last message as one standalone search query, using the conversation for "
                  "context. Also give two alternative phrasings. Do not answer the question.")


def rewriter_server(url: str, body: dict) -> dict:
    last = body["messages"][-1]["content"]
    scripted = {"And what if I live in Germany?": Rewrite(
        standalone_query="EU customer right to return opened items",
        alternatives=["withdrawal period for EU purchases", "refund rules for customers in Europe"])}
    out = scripted.get(last.split("</conversation>")[-1].strip())
    return claude_reply([{"type": "text", "text": out.model_dump_json()}])


def rewrite(llm: anthropic.Anthropic, history: list[str], last: str) -> Rewrite:
    convo = "\n".join(history)
    r = llm.messages.parse(model=FAST_MODEL, max_tokens=300, system=REWRITE_SYSTEM, output_format=Rewrite,
                           messages=[{"role": "user", "content": f"<conversation>{convo}</conversation>\n{last}"}])
    return r.parsed_output


eu_user = User(region="EU", groups={"public"})
history = ["user: I opened a stove and don't want it. Can I return it?", "assistant: Opened items: 14 days, store credit."]
last = "And what if I live in Germany?"
rw = rewrite(fake_claude(rewriter_server), history, last)
print("raw query       ->", [c.id for c in hybrid_search(dense, bm25, last, k=3, where=allowed_for(eu_user))])
print("rewritten       ->", [c.id for c in hybrid_search(dense, bm25, rw.standalone_query, k=3, where=allowed_for(eu_user))])
multi = rrf([[c.id for c in hybrid_search(dense, bm25, q, k=10, where=allowed_for(eu_user))]
             for q in [rw.standalone_query, *rw.alternatives]])
print("multi-query RRF ->", multi[:3])

> **Observe:** "And what if I live in Germany?" on its own retrieves the delivery-times section, because nothing in it is about returns; the standalone rewrite finds the EU withdrawal section. Multi-query fusion runs each phrasing and fuses the lists with the same `rrf()`.

## 6.10 Contextual retrieval

For every chunk, a fast model reads the whole document plus the chunk and writes a short context that is **prepended before indexing**, in both the dense and the BM25 index. The document goes first with `cache_control`, so every chunk after the first can read it from the cache. The fake server plays Claude Haiku 4.5: it finds the chunk inside the document and names the document and section, as the real model would. [Theory](README.md#610-contextual-retrieval)

In [ ]:
CONTEXT_PROMPT = ("Here is a chunk from the document above:\n<chunk>{chunk}</chunk>\nWrite one or two sentences that "
                  "situate this chunk within the document, to improve search retrieval. "
                  "Answer with only those sentences.")
MIN_CACHEABLE_HAIKU = 4096        # minimum cacheable prompt length on Claude Haiku 4.5, in tokens
_cache: set[str] = set()
CACHE_STATS = Counter()


def contextualizer_server(url: str, body: dict) -> dict:
    doc_block, prompt_block = body["messages"][0]["content"]
    doc = doc_block["text"].removeprefix("<document>").removesuffix("</document>")
    chunk = re.search(r"<chunk>(.*)</chunk>", prompt_block["text"], re.S).group(1)
    before = doc[: doc.find(chunk)]
    title, section = doc.splitlines()[0].lstrip("# "), re.findall(r"^## (.+)$", before, re.M)[-1]
    tokens = len(doc_block["text"]) // 4
    cacheable = "cache_control" in doc_block and tokens >= MIN_CACHEABLE_HAIKU
    hit = cacheable and doc in _cache
    if cacheable:
        _cache.add(doc)
    CACHE_STATS["read" if hit else "write" if cacheable else "uncached"] += tokens
    usage = {"input_tokens": len(prompt_block["text"]) // 4 + (0 if cacheable else tokens), "output_tokens": 25,
             "cache_creation_input_tokens": tokens if cacheable and not hit else 0,
             "cache_read_input_tokens": tokens if hit else 0}
    return claude_reply([{"type": "text", "text": f"From the {title}, section '{section}'."}], usage)


def contextualize(llm: anthropic.Anthropic, doc_text: str, chunk: Chunk) -> str:
    r = llm.messages.create(model=FAST_MODEL, max_tokens=150, messages=[{"role": "user", "content": [
        {"type": "text", "text": f"<document>{doc_text}</document>", "cache_control": {"type": "ephemeral"}},
        {"type": "text", "text": CONTEXT_PROMPT.format(chunk=chunk.text)}]}])
    return "".join(b.text for b in r.content if b.type == "text").strip()


CALLS.clear()
haiku = fake_claude(contextualizer_server)
for c in CHUNKS:
    c.meta["context"] = contextualize(haiku, DOCS[c.doc_id]["text"], c)
print(len(CALLS), "contextualizer calls; example:", BY_ID["summit-40#warranty-0"].meta["context"])
print(f"cache: {CACHE_STATS['write']} doc tokens written, {CACHE_STATS['read']} read, "
      f"{CACHE_STATS['uncached']} sent uncached (every doc here is below the {MIN_CACHEABLE_HAIKU}-token minimum)")

In [ ]:
def text_contextual(c: Chunk) -> str:
    return f"{c.meta['context']}\n{c.text}"


dense_ctx = DenseIndex(CHUNKS, ToyEmbedder(), text_of=text_contextual)
bm25_ctx = BM25(CHUNKS, text_of=text_contextual)


def contextual_rerank(question: str, where=PUBLIC_US) -> list[Chunk]:
    candidates = hybrid_search(dense_ctx, bm25_ctx, question, k=20, where=where)
    return [c for c, _ in toy_rerank(question, candidates, text_of=text_contextual)]


report("contextual hybrid + rerank", contextual_rerank)
print()
for name, r in RESULTS.items():
    print(f"{name:28} recall@1={r['recall@1']:.2f}  recall@3={r['recall@3']:.2f}  mrr={r['mrr']:.2f}")

In [ ]:
# Cost of contextual retrieval at Claude Haiku 4.5 list prices (USD per million tokens, checked 2026-10)
P_IN, P_WRITE, P_READ, P_OUT = 1.00, 1.25, 0.10, 5.00


def context_cost(doc_tokens: int, n_chunks: int, chunk_tokens: int, ctx_tokens: int = 100, prompt: int = 50,
                 cached: bool = True) -> float:
    doc = (doc_tokens * P_WRITE + (n_chunks - 1) * doc_tokens * P_READ) if cached else n_chunks * doc_tokens * P_IN
    return (doc + n_chunks * (chunk_tokens + prompt) * P_IN + n_chunks * ctx_tokens * P_OUT) / 1e6


with_cache, without = context_cost(8_000, 10, 800), context_cost(8_000, 10, 800, cached=False)
print(f"8k-token document, 10 chunks: ${with_cache:.4f} with caching vs ${without:.4f} without")
print(f"per million document tokens: ~${with_cache * 125:.2f} (cached), ~${with_cache * 125 / 2:.2f} via Message Batches")

In [ ]:
# (live, costs money: under $0.01) one real contextualizer call on Claude Haiku 4.5
if live("ANTHROPIC_API_KEY"):
    c = BY_ID["summit-40#warranty-0"]
    print(contextualize(anthropic.Anthropic(), DOCS[c.doc_id]["text"], c))

> **Observe:** the Summit 40 question is now answered at rank 1, because "Summit 40 backpack" is in the indexed text of the right chunk. In the cost cell, caching changes the bill only when documents are long and have many chunks; on Haiku 4.5, documents under 4,096 tokens are not cached at all, so short documents simply pay full input price.

## 6.11 Grounded generation with citations

The same `generate(question, hits)` interface for both vendors:

- **Claude:** one `search_result` block per chunk, citations enabled. The fake server answers from the first result and returns a real `search_result_location` citation, which the SDK parses into typed objects.
- **OpenAI:** numbered sources in the input and `[S1]`-style markers, checked in code.

A gate runs **before** either model: if the reranker's best score is below a threshold, return "not covered" without a model call. [Theory](README.md#611-grounded-generation-with-citations)

In [ ]:
GROUNDED = ("Answer only from the provided sources. If they do not contain the answer, say you cannot find it "
            "in the documents. Never use general knowledge. Keep it to two sentences.")


def first_sentence(text: str) -> str:
    return re.match(r"(.+?[.!?])(\s|$)", text).group(1)


@dataclass
class Answer:
    text: str
    sources: list[str]
    vendor: str


def claude_server(url: str, body: dict) -> dict:
    results = [b for b in body["messages"][0]["content"] if b["type"] == "search_result"]
    first = results[0]
    cited = first["content"][0]["text"]
    return claude_reply([{"type": "text", "text": first_sentence(cited), "citations": [
        {"type": "search_result_location", "cited_text": cited, "source": first["source"], "title": first["title"],
         "search_result_index": 0, "start_block_index": 0, "end_block_index": 1}]}])


class ClaudeGenerator:
    def __init__(self, client: anthropic.Anthropic, model: str = CLAUDE_MODEL):
        self.client, self.model = client, model

    def generate(self, question: str, hits: list[Chunk]) -> Answer:
        blocks = [{"type": "search_result", "source": c.meta["source"],
                   "title": f"{c.meta['title']} — {c.meta['section']}",
                   "content": [{"type": "text", "text": c.text}], "citations": {"enabled": True}} for c in hits]
        r = self.client.messages.create(model=self.model, max_tokens=1024, system=GROUNDED, messages=[
            {"role": "user", "content": blocks + [{"type": "text", "text": question}]}])
        text = "".join(b.text for b in r.content if b.type == "text")
        sources = sorted({cit.source for b in r.content if b.type == "text" for cit in (b.citations or [])})
        return Answer(text, sources, "claude")


hits = contextual_rerank("Can I cook inside my tent with the stove?")[:3]
ans = ClaudeGenerator(fake_claude(claude_server)).generate("Can I cook inside my tent with the stove?", hits)
print(ans)

In [ ]:
def openai_server(url: str, body: dict) -> dict:
    first = re.search(r"\[S1\][^\n]*\n(.+)", body["input"]).group(1)
    text = first_sentence(first) + " [S1]"
    return {"id": f"resp_{next(_ids):04d}", "object": "response", "created_at": 1_790_000_000, "model": body["model"],
            "status": "completed", "parallel_tool_calls": True, "tool_choice": "auto", "tools": [],
            "output": [{"type": "message", "id": "msg_1", "role": "assistant", "status": "completed",
                        "content": [{"type": "output_text", "text": text, "annotations": []}]}],
            "usage": {"input_tokens": 120, "output_tokens": 20, "total_tokens": 140,
                      "input_tokens_details": {"cached_tokens": 0}, "output_tokens_details": {"reasoning_tokens": 0}}}


class OpenAIGenerator:
    def __init__(self, client: openai.OpenAI, model: str = OPENAI_MODEL):
        self.client, self.model = client, model

    def generate(self, question: str, hits: list[Chunk]) -> Answer:
        numbered = "\n\n".join(f"[S{i}] {c.meta['title']} — {c.meta['section']}\n{c.text}"
                               for i, c in enumerate(hits, 1))
        r = self.client.responses.create(
            model=self.model, instructions=GROUNDED + " Cite sources as [S1], [S2].",
            input=f"<sources>\n{numbered}\n</sources>\n\nQuestion: {question}")
        markers = {int(n) for n in re.findall(r"\[S(\d+)\]", r.output_text)}
        if not markers or not markers <= set(range(1, len(hits) + 1)):
            raise ValueError(f"answer cites missing sources: {markers}")     # check markers in code
        return Answer(r.output_text, [hits[n - 1].meta["source"] for n in sorted(markers)], "openai")


print(OpenAIGenerator(fake_openai(openai_server)).generate("Can I cook inside my tent with the stove?", hits))

## 6.12 Managed File Search or a custom pipeline

With OpenAI's managed `file_search`, everything from 6.2 to 6.10 collapses into four calls: create a vector store, upload files with **attributes**, ask with the `file_search` tool, inspect what was retrieved. The live cell runs that flow on the same corpus, then **deletes the store** (storage is billed per GB-day). Compare its top results with your custom pipeline's. [Theory](README.md#612-managed-file-search-or-a-custom-pipeline)

In [ ]:
# (live, costs money: about $0.01) the same corpus in an OpenAI vector store with file_search
if live("OPENAI_API_KEY"):
    import io
    oai = openai.OpenAI()
    store = oai.vector_stores.create(name="ch06-practice", expires_after={"anchor": "last_active_at", "days": 1})
    try:
        for d in DOCS.values():
            m = d["meta"]
            oai.vector_stores.files.upload_and_poll(
                vector_store_id=store.id, file=(f"{d['doc_id']}.md", io.BytesIO(d["text"].encode())),
                attributes={"region": m["region"], "group": m["group"], "is_current": m["is_current"]})
        policy = {"type": "and", "filters": [{"type": "eq", "key": "group", "value": "public"},
                                             {"type": "eq", "key": "is_current", "value": True}]}
        r = oai.responses.create(
            model=OPENAI_MODEL, input="How long is the Summit 40 frame covered?",
            tools=[{"type": "file_search", "vector_store_ids": [store.id], "max_num_results": 5, "filters": policy}],
            include=["file_search_call.results"])
        print(r.output_text)
        for item in r.output:
            if item.type == "file_search_call":
                print("queries:", item.queries)
                for res in item.results or []:
                    print(f"  {res.score:.3f} {res.filename}")
    finally:
        oai.vector_stores.delete(store.id)

## Mini-project: one RAG pipeline, two vendors

Put the stages together behind one small interface:

- `RagPipeline(config)` builds the indexes once (plain or contextual text) and exposes `retrieve()` and `answer()`.
- `config` switches stages on and off, so you can measure what each one buys.
- `answer()` applies the permission filter inside retrieval, the "not covered" gate on the reranker score, then calls **any** generator with a `generate(question, hits)` method.

In [ ]:
@dataclass
class Config:
    hybrid: bool = True
    rerank: bool = True
    contextual: bool = True
    top_k: int = 3
    min_score: float = 0.3           # "not covered" gate on the reranker score; tune it on labeled questions


class RagPipeline:
    def __init__(self, chunks: list[Chunk], config: Config):
        self.cfg = config
        self.text_of = text_contextual if config.contextual else text_plain
        self.dense = DenseIndex(chunks, ToyEmbedder(), text_of=self.text_of)
        self.bm25 = BM25(chunks, text_of=self.text_of)

    def retrieve(self, question: str, user: User) -> list[tuple[Chunk, float]]:
        where = allowed_for(user)
        if self.cfg.hybrid:
            candidates = hybrid_search(self.dense, self.bm25, question, k=20, where=where)
        else:
            candidates = self.dense.search(question, k=20, where=where)
        if self.cfg.rerank:
            return toy_rerank(question, candidates, top_k=self.cfg.top_k, text_of=self.text_of)
        return [(c, 1.0) for c in candidates[: self.cfg.top_k]]

    def answer(self, question: str, user: User, generator) -> Answer:
        scored = self.retrieve(question, user)
        if not scored or scored[0][1] < self.cfg.min_score:
            return Answer("I can't find this in the documents you have access to.", [], "gate")
        return generator.generate(question, [c for c, _ in scored])


CONFIGS = {
    "dense only": Config(hybrid=False, rerank=False, contextual=False),
    "hybrid": Config(rerank=False, contextual=False),
    "hybrid + rerank": Config(contextual=False),
    "contextual hybrid + rerank": Config(),
}
for name, cfg in CONFIGS.items():
    pipe = RagPipeline(CHUNKS, cfg)
    r = evaluate(lambda q: [c for c, _ in pipe.retrieve(q, customer_us)])
    print(f"{name:28} recall@1={r['recall@1']:.2f} recall@3={r['recall@3']:.2f} mrr={r['mrr']:.2f}")

In [ ]:
pipe = RagPipeline(CHUNKS, Config())
generators = {"claude": ClaudeGenerator(fake_claude(claude_server)),
              "openai": OpenAIGenerator(fake_openai(openai_server))}
staff = User(region="US", groups={"public", "staff"})
cases = [("What does error E-305 mean on the Ember stove?", customer_us),
         ("What discount do employees get?", customer_us),      # internal doc: filtered out -> gate
         ("What discount do employees get?", staff),
         ("Do you sell kayaks?", customer_us)]                   # not in the corpus -> gate
for question, user in cases:
    top = pipe.retrieve(question, user)
    print(f"\nQ: {question}  (groups={sorted(user.groups)})  best score={top[0][1] if top else None}")
    for vendor, gen in generators.items():
        a = pipe.answer(question, user, gen)
        print(f"  {a.vendor:6} {a.text}  {a.sources}")

In [ ]:
# (live, costs money: about $0.01) the same pipeline with the real Claude and OpenAI models
if live("ANTHROPIC_API_KEY", "OPENAI_API_KEY"):
    real = {"claude": ClaudeGenerator(anthropic.Anthropic()), "openai": OpenAIGenerator(openai.OpenAI())}
    for vendor, gen in real.items():
        a = pipe.answer("Can I return an opened stove for a refund?", customer_us, gen)
        print(f"{vendor}: {a.text}\n   sources: {a.sources}")

> **Observe:** both vendors sit behind the same `generate()` method, so the retrieval stack, the permission filter and the gate are written once. The internal discount question is refused for a customer (the chunk never reaches the model) and answered for staff; the kayak question is refused because even the best reranked chunk scores below `min_score`.

> **Try it:**
> 1. Set `min_score=0.0` and rerun the cases: the kayak question now gets an answer built from an unrelated chunk. That is why the gate exists.
> 2. Ask "Is the tent waterproof?" (not in the corpus): the toy reranker scores the setup section about 0.46, so it passes the gate. A score gate is only a first filter; the generator's "say when the sources do not contain it" instruction is the second, and the threshold must be set on labeled answerable and unanswerable questions (chapter 07).
> 3. Add a question about a new product to `EVAL` and the corpus, re-run the config table, and check which stage it needs.
> 4. Swap `ToyEmbedder` for `OpenAIEmbedder(openai.OpenAI())` in `RagPipeline` (live) and compare the table.

## Recap

- **Ingestion decides the ceiling:** clean structure, metadata and content hashes are planned up front.
- **Structure-aware chunks** match the document's own units and set Claude's citation granularity.
- **Dense and BM25 fail differently**, so fuse them with RRF; **rerank** the fused candidates down to a few.
- **Filters run inside the search**, never in the prompt and never after top-k.
- **Contextual retrieval** fixes chunks that lost their document; caching only pays off on long, many-chunk documents.
- **Generate only from retrieved sources**, check citations in code, and gate "not covered" before the model call.

Next: [02_homework.ipynb](02_homework.ipynb)